# Compendium Data Pipeline

Loops through each domain folder (Health, Population, Education, Labor, Poverty,
Housing) under `datacollector_received_quest`, reshapes and merges every Excel
file's data/source tables, corrects the Arabic labels against
`translation dict.xlsx`, translates to English, and saves per-domain
`<Domain>_AR.xlsx` / `<Domain>_EN.xlsx` outputs to the COMPENDIUM-ARAB SOCIETY
folder.

Folder structure and conventions follow `test claude/compendium pipeline.md`.


In [ ]:
"""
CELL: Imports and logging setup.

This cell only sets things up — it does not touch any files. It imports the
libraries the rest of the notebook needs and configures a logger that prints
timestamped INFO / WARNING / ERROR lines for progress and failures.
(Dictionary replacement lines use plain print() instead, further down, so
they visually stand out from the progress log.)
"""
import difflib                      # fuzzy ("did you mean...") string matching
import logging                      # progress / warning / error messages
import re                           # whitespace normalization helper
from collections import defaultdict  # FAILURES: domain -> list of failure dicts
from pathlib import Path            # safe path joining (folder names contain spaces)

import pandas as pd                 # reading/reshaping/merging/writing Excel

# Every logger.info/warning/error(...) call below prints as:
# "HH:MM:SS | LEVEL | message"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium_pipeline")


## Config / paths

In [ ]:
"""
CELL: Configuration.

All folder paths, the list of domains to process, and the two tuning
constants (fuzzy-match cutoff, merge keys) live here. This is the cell to
edit if you want to run a subset of domains, or point the pipeline at a
different folder.
"""
# Root DATA COLLECTOR folder: holds both the raw questionnaires and the dictionary.
DATA_COLLECTOR_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
# Contains one subfolder per domain (Health, Population, ...), each full of .xlsx files.
RECEIVED_QUEST_PATH = DATA_COLLECTOR_PATH / "datacollector_received_quest"
# Single lookup file (col_en, val_en, col_ar, val_ar) used to standardize + translate every domain.
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
# Where the final <Domain>_AR.xlsx / <Domain>_EN.xlsx files get written.
OUTPUT_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")

# The six domain folders to loop over; each produces its own <name>_AR.xlsx / <name>_EN.xlsx.
DOMAINS = ["Health", "Population", "Education", "Labor", "Poverty", "Housing"]

# Minimum difflib similarity ratio (0-1) required to accept a fuzzy dictionary match.
# Below this, the original value is left untouched and a warning is logged,
# rather than force-replacing with a poor (potentially nonsensical) match.
FUZZY_MATCH_CUTOFF = 0.6

# Arabic column names that can be used to join the reshaped data table with the
# source table: year ("السنة"), indicator ("المؤشر"), country ("الدولة").
# Year alone is required; indicator/country are included only when both tables have them.
SOURCE_MERGE_KEYS_CANDIDATES = ["\u0627\u0644\u0633\u0646\u0629", "\u0627\u0644\u0645\u0624\u0634\u0631", "\u0627\u0644\u062f\u0648\u0644\u0629"]


## Load the translation dictionary

Loaded once per run (`translation dict.xlsx` has columns `col_en`, `val_en`,
`col_ar`, `val_ar`). Built into an Arabic-keyed lookup
`{ar_column: {"dim_values": {ar_value: en_value}, "dim": {ar_column: en_column}}}`
used for both the dictionary-correction step and the translation step.

In [ ]:
"""
CELL: Load & build the translation dictionary (runs once for the whole notebook).

Reads translation dict.xlsx and turns its four flat columns into two lookup
structures used everywhere else in the pipeline:
  - AR_EN_DICTIONARY: for every Arabic column name, the set of valid Arabic
    values in that column and what each one translates to in English.
  - CHAPTER_TO_ARABIC: English chapter name (e.g. "Labor") -> the Arabic
    "Chapter" label the raw sheets use for it (e.g. "\u0639\u0645\u0627\u0644\u0629").
"""


def load_translation_dictionary(path: Path) -> pd.DataFrame:
    """Reads translation dict.xlsx and checks it has the 4 expected columns."""
    logger.info(f"Loading translation dictionary from {path}")
    df = pd.read_excel(path, engine="openpyxl")
    required_cols = {"col_en", "val_en", "col_ar", "val_ar"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"translation dict.xlsx is missing expected columns: {missing}")
    return df


def build_dictionaries(df_translation: pd.DataFrame):
    """Returns (ar_en_dictionary, chapter_to_arabic).

    ar_en_dictionary: {ar_column: {"dim_values": {ar_value: en_value}, "dim": {ar_column: en_column}}}
    chapter_to_arabic: {English chapter name: Arabic chapter label}, e.g. {"Labor": "\u0639\u0645\u0627\u0644\u0629"}
    """
    ar_en_dictionary = {}
    # One dictionary "dimension" per distinct Arabic column name (e.g. الدولة/Country,
    # الجنس/Sex, ...). Each dimension maps its Arabic values to English values, plus the
    # column-name translation itself.
    for dim in df_translation["col_ar"].dropna().unique():
        df_dim = df_translation[df_translation["col_ar"] == dim]
        ar_en_dictionary[dim] = {
            "dim_values": dict(zip(df_dim["val_ar"], df_dim["val_en"])),
            "dim": {dim: df_dim["col_en"].iloc[0]},
        }

    # The dictionary also carries a "Chapter" dimension (col_en == "Chapter") whose
    # val_en/val_ar pairs are exactly the chapter names, e.g. "Labor" <-> "عمالة".
    # Reused directly to stamp each row with which chapter it came from.
    chapter_rows = df_translation[df_translation["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return ar_en_dictionary, chapter_to_arabic


# Load once here; every downstream function receives these as arguments rather
# than re-reading the Excel file per domain/file.
df_translation = load_translation_dictionary(TRANSLATION_DICT_PATH)
AR_EN_DICTIONARY, CHAPTER_TO_ARABIC = build_dictionaries(df_translation)
logger.info(f"Dictionary loaded: {len(AR_EN_DICTIONARY)} Arabic columns available for lookups")

# Sanity check: every domain we plan to process must have a Chapter mapping,
# otherwise we would not know what to stamp its rows with.
missing_domains = [d for d in DOMAINS if d not in CHAPTER_TO_ARABIC]
if missing_domains:
    logger.warning(f"No 'Chapter' entry in translation dict.xlsx for: {missing_domains}")


## `extract_tables()`

Each sheet has two tables marked by an `index` column: `index=1` is the data
table, `index=2` is the source table. Both tables share the same header row
convention: a row where column 0 equals the literal string `"index"` holds
the column names for the block that follows it.

In [ ]:
"""
CELL: extract_tables() — split one raw sheet into its data table and source table.

Every questionnaire sheet is laid out as two side-by-side mini-tables stacked
in the same grid: rows where column 0 == "1" are the actual data (values per
year), rows where column 0 == "2" are the source/citation for that data. Each
mini-table has its own header row, marked by column 0 == "index". This
function locates those two header rows and slices out two clean DataFrames.
"""


def extract_tables(df_raw: pd.DataFrame, sheet_name: str):
    """Split a raw sheet (read with header=None, dtype=str) into the data
    table (index=1) and the source table (index=2)."""
    if df_raw.empty or df_raw.shape[1] == 0:
        raise ValueError("sheet is empty")

    # Column 0 says "index" on exactly the two header rows: first the data
    # table's header, then (further down) the source table's header.
    header_rows = df_raw.index[df_raw[0] == "index"].tolist()
    if len(header_rows) < 2:
        raise ValueError(
            f"expected 2 'index' header rows (data + source), found {len(header_rows)}"
        )
    data_header_row, source_header_row = header_rows[0], header_rows[1]

    # --- Data table (index == "1") ---
    # dropna() on the header row handles sheets where the data table's header
    # doesn't span every column in the sheet (the source table's columns
    # start further along the same rows).
    data_cols = df_raw.iloc[data_header_row].dropna().astype(str).str.strip()
    df_data = df_raw[df_raw[0] == "1"].copy()
    if df_data.empty:
        raise ValueError("no rows found with index=1 (data table)")
    df_data = df_data[data_cols.index]   # keep only the columns under this header
    df_data.columns = data_cols.values   # apply the real (Arabic) column names
    df_data = df_data.drop(columns=["index"], errors="ignore")

    # --- Source table (index == "2") --- same logic, its own header row.
    source_cols = df_raw.iloc[source_header_row].dropna().astype(str).str.strip()
    df_source = df_raw[df_raw[0] == "2"].copy()
    if df_source.empty:
        raise ValueError("no rows found with index=2 (source table)")
    df_source = df_source[source_cols.index]
    df_source.columns = source_cols.values
    df_source = df_source.drop(columns=["index"], errors="ignore")

    return df_data, df_source


## `reshape_and_merge()`

Unpivots the data table's year columns (any column whose name is all
digits) from wide to long format, keeping every other column as an
identifier, then merges in the matching source-table row on year (+
indicator/country, when present) so each data row carries its source.

Raises a `PipelineStepError` tagged `"reshape"` or `"merge"` so the calling
loop can report precisely which half failed.

In [ ]:
"""
CELL: reshape_and_merge() — wide-to-long reshape, then merge in the source.

Data tables are wide: one row per (indicator, country, sex, ...) combination,
with one column per year (2010, 2011, ...). This melts those year columns
down into two columns (year, value), then merges in the matching row from the
source table so every (indicator, country, year) data point carries its
citation. Wrapped so a failure in either half is reported with a precise
step name ("reshape" vs "merge") by the caller.
"""


class PipelineStepError(Exception):
    """Wraps an exception with the pipeline step name it occurred in, so the
    main loop's error log can say exactly which sub-step failed."""

    def __init__(self, step: str, original_exc: Exception):
        self.step = step
        self.original_exc = original_exc
        super().__init__(str(original_exc))


def reshape_and_merge(df_data: pd.DataFrame, df_source: pd.DataFrame) -> pd.DataFrame:
    # --- Reshape: wide (one column per year) -> long (one row per year) ---
    try:
        # Year columns are the ones named with plain digits (e.g. "2010");
        # everything else (Indicator, Country, Sex, ...) is an identifier
        # column that gets repeated for every year row.
        id_vars = [c for c in df_data.columns if not str(c).isdigit()]
        year_columns = [c for c in df_data.columns if str(c).isdigit()]
        if not year_columns:
            raise ValueError("no year columns found in the data table (expected 4-digit column headers)")

        df_melted = df_data.melt(
            id_vars=id_vars,
            value_vars=year_columns,
            var_name="\u0627\u0644\u0633\u0646\u0629",   # "Year"
            value_name="\u0627\u0644\u0639\u062f\u062f",  # "Value"
        )
    except Exception as exc:
        raise PipelineStepError("reshape", exc) from exc

    # --- Merge: attach each row's source/citation from the source table ---
    try:
        # Only join on keys that actually exist in both tables. Year is
        # mandatory; Indicator/Country are added when the source table
        # carries them too (some domains' source tables are just year+source).
        merge_keys = [
            key for key in SOURCE_MERGE_KEYS_CANDIDATES
            if key in df_melted.columns and key in df_source.columns
        ]
        if "\u0627\u0644\u0633\u0646\u0629" not in merge_keys:
            raise KeyError("year column missing from data or source table, cannot merge")
        df_merged = pd.merge(df_melted, df_source, on=merge_keys, how="left")
    except Exception as exc:
        raise PipelineStepError("merge", exc) from exc

    return df_merged


## `correct_with_dictionary()`

For every column name and every cell value: if it (after whitespace
normalization) exactly matches an entry in the dictionary, it is
standardized to that entry. Otherwise, a similarity score is computed
against every dictionary entry for that column, and the value is replaced
with the highest-scoring match — but only if that score clears
`FUZZY_MATCH_CUTOFF`; below the cutoff the original value is left as-is and
a warning is logged, rather than forcing in a poor match. Every actual
replacement is printed.

In [ ]:
"""
CELL: correct_with_dictionary() — standardize column names and cell values.

Raw questionnaire text often has small variations from the dictionary's
canonical spelling (extra spaces, a trailing typo, a slightly different
wording). This cell fixes column names and values in place: an exact match
(after normalizing whitespace) is swapped for the dictionary's canonical
spelling; anything else is fuzzy-matched (difflib) against every candidate
for that column and swapped for the best match, but only above
FUZZY_MATCH_CUTOFF so a bad guess isn't forced in. Every actual change is
printed with print(), as requested, so the corrections are visible in the
notebook output.
"""


def _normalize(value) -> str:
    """Collapses any run of whitespace (including non-breaking spaces) to a
    single regular space, and strips the ends. Used only for comparisons —
    the dictionary's own (un-normalized) spelling is always what gets
    substituted in."""
    return re.sub(r"\s+", " ", str(value).strip())


def _best_fuzzy_match(value: str, candidates):
    """Returns (best_candidate, score) — the dictionary entry with the
    highest difflib similarity ratio against value. Always returns the
    single best candidate; the caller decides whether the score is good
    enough to act on."""
    best_candidate, best_score = None, -1.0
    for candidate in candidates:
        score = difflib.SequenceMatcher(None, value, candidate).ratio()
        if score > best_score:
            best_candidate, best_score = candidate, score
    return best_candidate, best_score


def correct_with_dictionary(df: pd.DataFrame, ar_en_dictionary: dict, domain: str, file_name: str, sheet_name: str):
    """Standardizes column names and cell values against ar_en_dictionary.
    Returns (corrected_df, replacement_count).

    Comparisons are done on normalized strings on both sides (so e.g. a
    non-breaking space vs a regular space counts as an exact dictionary hit,
    not a fuzzy match), but replacements always substitute in the
    dictionary's own canonical (raw, un-normalized) spelling.
    """
    df_corrected = df.copy()
    # normalized dictionary column name -> its real (canonical) spelling
    known_columns_norm = {_normalize(c): c for c in ar_en_dictionary.keys()}
    context = f"[{domain}] {file_name} | {sheet_name}"  # prefix for every printed line
    replacements = 0

    # --- 1. Column name correction ---
    for col in list(df_corrected.columns):
        col_str = str(col)
        normalized = _normalize(col_str)
        if normalized in known_columns_norm:
            # Exact dictionary hit (post-normalization). Only log/rename if the
            # raw spelling actually differs from the dictionary's canonical one.
            canonical = known_columns_norm[normalized]
            if canonical != col_str:
                print(f"{context} | COLUMN: {col_str!r} -> {canonical!r}")
                df_corrected.rename(columns={col: canonical}, inplace=True)
                replacements += 1
            continue
        # Not an exact hit: find the closest dictionary column name and only
        # accept it if the similarity score clears the configured cutoff.
        best_match, score = _best_fuzzy_match(normalized, known_columns_norm.keys())
        if best_match is not None and score >= FUZZY_MATCH_CUTOFF:
            canonical = known_columns_norm[best_match]
            print(f"{context} | COLUMN: {col_str!r} -> {canonical!r} (score={score:.2f})")
            df_corrected.rename(columns={col: canonical}, inplace=True)
            replacements += 1
        elif best_match is not None:
            logger.warning(
                f"{context} | COLUMN: no dictionary match for {col_str!r} "
                f"above cutoff (best={best_match!r}, score={score:.2f}) — left unchanged"
            )

    # --- 2. Cell value correction, per column that is a known dictionary dimension ---
    for col in df_corrected.columns:
        if col not in ar_en_dictionary:
            # Columns like Year/Value/Source (or anything not in the
            # dictionary) have no fixed vocabulary to check values against.
            continue
        allowed_norm = {
            _normalize(v): v for v in ar_en_dictionary[col]["dim_values"].keys() if pd.notna(v)
        }
        if not allowed_norm:
            # Some dictionary columns (Year/Value/Source) only rename the
            # column and carry no value vocabulary — nothing to check here.
            continue

        for val in df_corrected[col].dropna().unique():
            val_str = str(val)
            normalized = _normalize(val_str)
            if normalized in allowed_norm:
                canonical = allowed_norm[normalized]
                if canonical != val_str:
                    print(f"{context} | {col}: {val_str!r} -> {canonical!r}")
                    df_corrected[col] = df_corrected[col].replace(val, canonical)
                    replacements += 1
                continue
            best_match, score = _best_fuzzy_match(normalized, allowed_norm.keys())
            if best_match is not None and score >= FUZZY_MATCH_CUTOFF:
                canonical = allowed_norm[best_match]
                print(f"{context} | {col}: {val_str!r} -> {canonical!r} (score={score:.2f})")
                df_corrected[col] = df_corrected[col].replace(val, canonical)
                replacements += 1
            elif best_match is not None:
                logger.warning(
                    f"{context} | {col}: no dictionary match for {val_str!r} "
                    f"above cutoff (best={best_match!r}, score={score:.2f}) — left unchanged"
                )

    return df_corrected, replacements


## `translate()`

Renames columns and replaces cell values from Arabic to English using the same dictionary, now that values have been standardized.

In [ ]:
"""
CELL: translate() — Arabic to English, using the same dictionary.

Runs after correct_with_dictionary(), so column names and values are already
standardized to exactly what the dictionary expects. For every Arabic column
present in the dictionary, this replaces its values with their English
equivalents and renames the column to its English name.
"""


def translate(df: pd.DataFrame, ar_en_dictionary: dict) -> pd.DataFrame:
    df_translated = df.copy()
    for ar_col, col_dict in ar_en_dictionary.items():
        if ar_col in df_translated.columns:
            # Swap Arabic values -> English values, then rename the column itself.
            df_translated[ar_col] = df_translated[ar_col].replace(col_dict["dim_values"])
            df_translated.rename(columns=col_dict["dim"], inplace=True)
    return df_translated


## Error tracking

Every file/sheet is processed inside its own try/except so one bad file
never crashes the domain loop. Failures are logged immediately and also
collected so a full summary can be printed at the end of the run.

In [ ]:
"""
CELL: Error tracking helpers.

FAILURES collects every file/sheet failure (grouped by domain) so the final
cell can print one consolidated summary. log_failure() is called from inside
each try/except in process_domain() below; it both logs the error
immediately (with the domain/file/sheet/step/exception/likely-cause detail
requested) and records it in FAILURES for that end-of-run summary.
"""
FAILURES = defaultdict(list)  # domain -> list of {file, sheet, step, error}

# Plain-language guess at what's wrong, keyed by which processing step failed.
LIKELY_CAUSES = {
    "read": "file may be corrupted, password-protected, or not a valid .xlsx",
    "extract": "sheet may be missing an 'index' column, or index values are not 1/2 as expected",
    "reshape": "data table may be missing year columns, or id/year columns are malformed",
    "merge": "year/indicator/country columns may not match between the data and source tables",
    "dictionary correction": "column names or values may be malformed (e.g. all-blank) and unable to be matched",
    "translation": "a column or value may not have a corresponding English entry in the dictionary",
}


def log_failure(domain: str, file_name: str, sheet_name: str, step: str, exc: Exception):
    """Logs one [ERROR] line with full context, and records the failure so
    it also appears in the end-of-run summary."""
    likely_cause = LIKELY_CAUSES.get(step, "unexpected error, inspect the sheet manually")
    logger.error(
        f"[ERROR] Domain={domain} | File={file_name} | Sheet={sheet_name} | "
        f"Step={step} | {type(exc).__name__}: {exc} | Likely cause: {likely_cause}"
    )
    FAILURES[domain].append({
        "file": file_name,
        "sheet": sheet_name,
        "step": step,
        "error": f"{type(exc).__name__}: {exc}",
    })


## `process_domain()` — the per-domain orchestrator

For a single domain: finds every `.xlsx` file in its folder, processes every
sheet through extract -> reshape/merge -> dictionary correction -> translation,
appends the per-file results, then concatenates and saves `<Domain>_AR.xlsx`
(corrected, pre-translation) and `<Domain>_EN.xlsx` (translated).

In [ ]:
"""
CELL: process_domain() — ties every step together for one domain.

For a single domain folder: lists its .xlsx files, and for every sheet in
every file runs extract_tables() -> reshape_and_merge() ->
correct_with_dictionary() -> translate(), each wrapped in its own try/except
so one bad file/sheet is logged and skipped rather than crashing the whole
domain. Successful sheets are collected into two lists (Arabic-corrected,
English-translated), concatenated at the end, and saved as
<Domain>_AR.xlsx / <Domain>_EN.xlsx.
"""


def process_domain(domain: str, base_path: Path, output_path: Path, ar_en_dictionary: dict, chapter_to_arabic: dict):
    folder = base_path / domain
    if not folder.exists():
        logger.warning(f"Domain folder not found, skipping: {folder}")
        return
    if domain not in chapter_to_arabic:
        logger.warning(f"'{domain}' has no Chapter entry in the translation dictionary, skipping")
        return

    # Skip Excel's own temp/lock files (start with "~$") for files that are currently open.
    xlsx_files = sorted(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$"))
    logger.info(f"Processing {domain}: {len(xlsx_files)} file(s) found")

    ar_frames, en_frames = [], []  # one entry per successfully processed sheet
    total_replacements = 0

    for file_path in xlsx_files:
        file_name = file_path.name
        try:
            xls = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as exc:
            # Whole file unreadable (corrupt, wrong format, etc.) - skip the file, keep going.
            log_failure(domain, file_name, "-", "read", exc)
            continue

        logger.info(f"  {domain}/{file_name}: {len(xls.sheet_names)} sheet(s)")

        for sheet_name in xls.sheet_names:
            # Step 0: read the raw sheet as plain strings, no header inference
            # (extract_tables() locates the real header rows itself).
            try:
                df_raw = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)
            except Exception as exc:
                log_failure(domain, file_name, sheet_name, "read", exc)
                continue

            # Step 1: split into data table (index=1) and source table (index=2).
            try:
                df_data, df_source = extract_tables(df_raw, sheet_name)
            except Exception as exc:
                log_failure(domain, file_name, sheet_name, "extract", exc)
                continue

            # Stamp every row with which domain/chapter it belongs to, using the
            # domain we already know from the folder (not guessed from the sheet name).
            df_data["\u0627\u0644\u0641\u0635\u0644"] = chapter_to_arabic[domain]  # "Chapter"

            # Step 2: wide -> long reshape, then merge in the matching source rows.
            try:
                df_merged = reshape_and_merge(df_data, df_source)
            except PipelineStepError as exc:
                # exc.step is "reshape" or "merge" - reported precisely, per spec.
                log_failure(domain, file_name, sheet_name, exc.step, exc.original_exc)
                continue
            except Exception as exc:
                log_failure(domain, file_name, sheet_name, "reshape/merge", exc)
                continue

            # Step 3: standardize column names/values against the dictionary.
            try:
                df_corrected, n_repl = correct_with_dictionary(
                    df_merged, ar_en_dictionary, domain, file_name, sheet_name
                )
                total_replacements += n_repl
            except Exception as exc:
                log_failure(domain, file_name, sheet_name, "dictionary correction", exc)
                continue

            # Step 4: translate the now-standardized Arabic data to English.
            try:
                df_en = translate(df_corrected, ar_en_dictionary)
            except Exception as exc:
                log_failure(domain, file_name, sheet_name, "translation", exc)
                continue

            # Success: keep both the pre-translation (AR) and translated (EN) versions.
            ar_frames.append(df_corrected)
            en_frames.append(df_en)

    logger.info(
        f"{domain}: {len(ar_frames)} sheet(s) processed successfully, "
        f"{total_replacements} dictionary replacement(s) made"
    )

    if not ar_frames:
        logger.warning(f"{domain}: no data extracted, no output files written")
        return

    # Stack every sheet's rows on top of each other (concatenate, not merge side-by-side).
    df_ar = pd.concat(ar_frames, ignore_index=True)
    df_en = pd.concat(en_frames, ignore_index=True)

    ar_out = output_path / f"{domain}_AR.xlsx"
    en_out = output_path / f"{domain}_EN.xlsx"
    df_ar.to_excel(ar_out, index=False, engine="openpyxl")
    df_en.to_excel(en_out, index=False, engine="openpyxl")
    logger.info(f"{domain}: saved {ar_out.name} and {en_out.name}")


## Run the pipeline for all domains

In [ ]:
"""
CELL: Main run — calls process_domain() once per domain in DOMAINS.

This is the cell that actually touches the filesystem: it reads every Excel
file under each domain folder and writes the <Domain>_AR.xlsx /
<Domain>_EN.xlsx outputs. FAILURES is cleared first so re-running this cell
doesn't accumulate failures from a previous run.
"""
FAILURES.clear()

for domain in DOMAINS:
    process_domain(domain, RECEIVED_QUEST_PATH, OUTPUT_PATH, AR_EN_DICTIONARY, CHAPTER_TO_ARABIC)


## Run summary — failures grouped by domain

In [ ]:
"""
CELL: Run summary.

Prints every failure collected in FAILURES during the run above, grouped by
domain, so every file/sheet that needs attention is visible in one place
instead of scattered through the progress log.
"""
print("\n" + "=" * 70)
print("RUN SUMMARY - FAILURES BY DOMAIN")
print("=" * 70)

if not FAILURES:
    print("No failures. All files/sheets processed successfully.")
else:
    total = sum(len(v) for v in FAILURES.values())
    print(f"{total} failure(s) across {len(FAILURES)} domain(s):\n")
    for domain, failures in FAILURES.items():
        print(f"{domain} ({len(failures)} failure(s)):")
        for f in failures:
            print(f"  - {f['file']} | Sheet={f['sheet']} | Step={f['step']} | {f['error']}")
        print()
